# 웹 문서 기반 RAG 구축하기

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- `WebBaseLoader`로 웹 페이지를 크롤링해 Document 객체로 변환하기
- `BeautifulSoup`의 `SoupStrainer`로 HTML에서 필요한 영역만 추출하기
- 웹 문서의 특성(HTML 노이즈, 긴 텍스트)을 고려한 전처리 적용하기

## 사전 지식

- 00-RAG-Basic.ipynb의 8단계 RAG 파이프라인 이해
- HTML 기본 구조 (태그, 클래스, id 등)

---

```mermaid
flowchart LR
    URL[웹 페이지 URL]:::input --> WL[WebBaseLoader<br/>HTTP 요청]:::process
    WL --> BS[BeautifulSoup<br/>HTML 파싱]:::process
    BS --> SS[SoupStrainer<br/>본문 영역만 추출]:::process
    SS --> D[Document 객체]:::process
    D --> RAG[RAG 파이프라인<br/>분할 → 임베딩 → 검색]:::process
    RAG --> A[웹 기반<br/>질의응답]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## PDF vs 웹 문서

| 특성 | PDF | 웹 문서 |
|------|-----|---------|
| 업데이트 | 파일 교체 필요 | URL 그대로, 자동 반영 |
| 구조 | 고정된 레이아웃 | HTML 태그 파싱 필요 |
| 접근성 | 파일 배포 필요 | URL만 있으면 즉시 |
| 노이즈 | 적음 | 네비게이션·광고 제거 필요 |

> **실무 팁**: `SoupStrainer`에 Wikipedia의 `mw-parser-output` 같은 본문 클래스를 지정하면 불필요한 HTML 요소를 효과적으로 제거할 수 있어요.

## 학습 목표

- `WebBaseLoader`를 사용한 웹 페이지 크롤링
- BeautifulSoup을 활용한 HTML 파싱
- 웹 문서의 특성을 고려한 전처리
- 웹 기반 RAG 시스템 구축 및 테스트

## 웹 문서 vs PDF 문서

### 웹 문서의 특징

**장점**:
- ✅ 실시간 업데이트: 최신 정보 반영 가능
- ✅ 접근성: URL만 있으면 즉시 접근
- ✅ 구조화: HTML 태그로 구조 파악 용이

**주의사항**:
- ⚠️ HTML 노이즈: 네비게이션, 광고 등 불필요한 요소 제거 필요
- ⚠️ 동적 콘텐츠: JavaScript로 생성되는 내용은 추가 처리 필요
- ⚠️ 웹 크롤링 정책: robots.txt 및 이용 약관 확인 필수

### PDF 문서의 특징

**장점**:
- ✅ 안정성: 내용 변경 없음
- ✅ 완결성: 전체 문서가 하나의 파일
- ✅ 형식 보존: 레이아웃과 서식 유지

**주의사항**:
- ⚠️ 업데이트: 수동으로 파일 교체 필요
- ⚠️ 파싱 난이도: 복잡한 레이아웃 처리 어려움

## 환경 설정

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()

True

## 실습 웹사이트

이번 실습에서는 **Wikipedia 한국어 페이지**를 사용합니다.

- 주제: 인공지능 (Artificial Intelligence)
- URL: https://ko.wikipedia.org/wiki/인공지능
- 선택 이유: 안정적이고 구조화된 콘텐츠, 풍부한 정보

**학습 목표**: 웹 페이지를 크롤링하여 인공지능에 대한 질문에 답변하는 RAG 시스템 구축

## 필수 라이브러리 import

**웹 크롤링을 위한 추가 라이브러리**:
- `WebBaseLoader`: 웹 페이지 로딩
- `bs4` (BeautifulSoup): HTML 파싱 및 필터링

In [9]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [15]:
# ---------------------------------------------------
# WebBaseLoader로 웹 페이지 크롤링 — 본문 영역만 추출
# ---------------------------------------------------
# ============================================================
# TODO: WebBaseLoader와 SoupStrainer로 Wikipedia 본문을 로드하세요
# 힌트: bs_kwargs={"parse_only": bs4.SoupStrainer("div", attrs={...})}
# 예상 결과: 로드된 문서 수와 문자 길이 출력
# ============================================================

# 단계 1: 웹 페이지 로드
# URL="https://ko.wikipedia.org/wiki/인공지능"
loader = WebBaseLoader(
    web_paths=["https://ko.wikipedia.org/wiki/인공지능"],
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"id": "bodyContent"}  # :warning: 주의: 사이트마다 본문 클래스가 다름
        )
    )
)

docs = loader.load()

In [ ]:
# ---------------------------------------------------
# WebBaseLoader로 웹 페이지 크롤링 — 본문 영역만 추출
# ---------------------------------------------------
# ============================================================
# TODO: WebBaseLoader와 SoupStrainer로 Wikipedia 본문을 로드하세요
# 힌트: bs_kwargs={"parse_only": bs4.SoupStrainer("div", attrs={...})}
# 예상 결과: 로드된 문서 수와 문자 길이 출력
# ============================================================

# 단계 1: 웹 페이지 로드
# SoupStrainer: HTML 전체를 파싱하지 않고 지정 영역만 추출 (속도·메모리 절약)


웹 페이지의 메타데이터를 확인합니다. URL, 제목 등의 정보가 포함되어 있습니다.

In [ ]:
# 메타데이터 확인

# 아래에 코드를 작성하세요


### 2단계: 문서 분할 (Text Splitting)

웹 페이지는 일반적으로 하나의 긴 문서로 로드되므로 적절한 크기로 분할이 필요합니다.

**웹 문서 분할 시 고려사항**:
- 단락 구분이 명확하지 않을 수 있음
- HTML 태그 제거 후 공백 처리 필요
- 제목과 본문의 연결성 유지

In [35]:
# 단계 2: 문서 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

split_documents = text_splitter.split_documents(docs)

### 3-4단계: 임베딩 및 벡터 저장

PDF와 동일한 방식으로 임베딩 모델을 생성하고 벡터스토어에 저장합니다.

In [36]:
# ---------------------------------------------------
# OpenAI 임베딩으로 웹 문서를 FAISS 벡터스토어에 저장
# ---------------------------------------------------
# ============================================================
# TODO: split_documents를 임베딩하여 FAISS 벡터스토어를 만드세요
# 힌트: OpenAIEmbeddings(model="text-embedding-3-small"), FAISS.from_documents(...)
# 예상 결과: "벡터스토어 생성 완료" 출력
# ============================================================

# 단계 3: 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 단계 4: 벡터스토어 생성
vectorstore = FAISS.from_documents(
    embedding=embeddings,
    documents=split_documents
)

웹 문서에서 특정 키워드로 검색이 잘 되는지 테스트합니다.

In [37]:
# 벡터스토어 검색 테스트
test_query = "인공지능"
res = vectorstore.similarity_search(test_query, k=2)

res

[Document(id='f876c279-8f77-4cc6-a7de-0a2533e20785', metadata={'source': 'https://ko.wikipedia.org/wiki/인공지능'}, page_content='인공지능은 대규모 데이터를 수집·분석하는 과정에서 개인의 민감한 정보를 처리하게 된다. 이때 데이터 관리가 부실하거나 보안이 취약할 경우, 개인정보 유출 및 프라이버시 침해의 위험이 높아진다. 특히 얼굴 인식 기술이나 음성 비서 서비스 등은 사용자의 생체 정보를 기반으로 하기 때문에, 기술 오용 시 심각한 피해를 초래할 수 있다.\n사고력 저하\n인공지능에 대한 과도한 의존은 인간의 사고력 저하로 이어질 수 있다. AI가 대부분의 판단과 결정을 대신하게 되면, 인간은 스스로 사고하고 문제를 해결하는 능력이 감소할 가능성이 있다. 이는 학습 능력과 비판적 사고, 창의성의 저하로 이어질 수 있다는 지적이 있다.\n인공지능 기술의 실용적인 응용\n인공지능의 궁극적인 목표인 인간과 같은 지능의 개발이 어려움을 겪자, 다양한 응용 분야가 나타나게 되었다. 대표적인 예가 LISP나 Prolog와 같은 언어인데, 애초에 인공지능 연구를 위해 만들어졌으나 지금에 와서는 인공지능과 관련이 없는 분야에서도 사용되고 있다. 해커 문화도 인공지능 연구실에서 만들어졌는데, 이 중에서도 다양한 시기에 매카시, 민스키, 페퍼트, 위노그라드(SHRDLU를 만든 뒤에 인공지능을 포기했다)와 같은 유명인의 모태가 된 MIT 인공지능 연구소가 유명하다.\n다른 많은 시스템들이 한때 인공지능의 활발한 연구 주제였던 기술들에 바탕을 두고 만들어졌다. 그 예들은 다음과 같다:\n체커스 게임에서 Chinook은 사람과 기계를 통합한 세계 챔피언을 차지했다. (1994년)\n체스를 두는 컴퓨터인 딥 블루(Deep Blue)의 성능 향상 버전(비공식적 명칭: 디퍼 블루(Deeper Blue)이 당시 세계 체스 챔피언 가리 카스파로프를 물리쳤다. (1997년)\n불확실한 상황에서 추론

### 5-8단계: 검색기, 프롬프트, LLM, 체인 생성

이후 단계는 PDF 기반 RAG와 동일합니다. 검색기를 생성하고 RAG 체인을 구축합니다.

In [52]:
# 단계 5: 검색기 생성

# retriever = vectorstore.as_retriever(
#     search_type="similarity",
#     search_kwargs={"k": 4}
# )

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 4,
        "score_threshold": 0.4
    }
)



In [46]:
# 단계 6: 프롬프트 템플릿

prompt = PromptTemplate.from_template("""
당신은 인터넷 문서 기반으로 질의응답을 수행하는 AI 어시스턴트입니다.
주어진 문맥(Context)를 바탕으로 질문(Question)을 답변해 주세요.

답변시 주의사항
    - 문맥에서 찾을 수 있는 정보만 사용하세요.
    - 문맥에 없는 내용이라면 "주어진 정보를 찾을 수 없습니다"라고 답하세요.
    - 답변은 명확하고 간결하게 작성하세요.
    - 한국어로 답변하세요.

#Context:
{context}
#Question:
{question}
#Answer:
""")

In [47]:
# ---------------------------------------------------
# LLM 생성
# ---------------------------------------------------

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


In [54]:
# ---------------------------------------------------
# LCEL로 RAG 체인 조립 — 검색기 + 프롬프트 + LLM 연결
# ---------------------------------------------------
# ============================================================
# TODO: retriever, prompt, llm을 LCEL 파이프라인으로 연결하세요
# 힌트: {"context": retriever, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
# 예상 결과: "RAG 체인 생성 완료" 출력
# ============================================================

# 단계 8: RAG 체인 생성
# RunnablePassthrough(): 사용자 질문을 그대로 "question" 키로 전달

# chain = (
#     {
#         "context": retriever,
#         "question": RunnablePassthrough()
#     }
#     | prompt
#     | llm
#     | StrOutputParser()
# )
from operator import itemgetter
from langchain_core.runnables import RunnableParallel



rag_chain = (
    RunnableParallel({"question": RunnablePassthrough()}).assign(context=itemgetter("question") | retriever) 
    | prompt
    | llm
    | StrOutputParser()
)



## 웹 기반 RAG 실행 및 테스트

이제 완성된 RAG 체인을 사용하여 인공지능 Wikipedia 페이지에 대한 질문에 답변해봅니다.

In [55]:
# 질문 1: 기본 개념 질의

query = "인공지능이란 무엇인지 기본 개념을 설명해주세요."
res = rag_chain.invoke(query)

res

'인공지능(Artificial Intelligence, AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 컴퓨터 과학의 세부분야입니다. 이는 자연 지능과는 다른 개념으로, 인간의 지능을 모방한 기능을 갖춘 컴퓨터 시스템을 의미합니다. 인공지능은 일반적으로 범용 컴퓨터에 적용되며, 이러한 지능을 만들 수 있는 방법론이나 실현 가능성을 연구하는 과학 기술 분야를 포함합니다.'

In [50]:
# 질문 2: 구체적인 정보 질의

query = "인공지능이란 무엇인지 구체적인 정보를 설명해주세요."
res = rag_chain.invoke(query)

res

No relevant docs were retrieved using the relevance score threshold 0.8


'주어진 정보를 찾을 수 없습니다.'

In [51]:
# 질문 3: 역사적 정보 질의

query = "인공지능이의 역사적 정보를 설명해주세요."
res = rag_chain.invoke(query)

res


No relevant docs were retrieved using the relevance score threshold 0.8


'주어진 정보를 찾을 수 없습니다.'

## 전체 코드 (통합 버전)

In [ ]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

    # 1. 웹 페이지 로드

    # 2. 문서 분할

        # 3. 임베딩 생성

        # 4. 벡터스토어 생성

        # 5. 검색기 생성

        # 6. 프롬프트


#Context:

#Question:

#Answer:

        # 7. LLM

        # 8. RAG 체인

        # 실행

# 아래에 코드를 작성하세요


> **실무 팁**: 웹 크롤링 기반 RAG를 프로덕션에 적용할 때는 크롤링 결과를 캐시(파일 또는 DB)에 저장하세요. 매 요청마다 웹을 크롤링하면 느리고 외부 사이트 장애에 취약합니다. 주기적 크롤링 + 벡터스토어 갱신 패턴을 권장합니다.

## 💡 핵심 정리

### 웹 기반 RAG의 특징

1. **WebBaseLoader**: URL만으로 간단히 문서 로드
2. **BeautifulSoup**: HTML 파싱으로 필요한 부분만 추출
3. **실시간성**: 웹 페이지 업데이트 시 즉시 반영 가능
4. **접근성**: 파일 다운로드 없이 URL로 접근

### PDF vs 웹 선택 가이드

| 상황 | 추천 소스 | 이유 |
|------|----------|------|
| **최신 정보 필요** | 웹 | 실시간 업데이트 반영 |
| **안정적인 참조** | PDF | 내용 변경 없음 |
| **빠른 프로토타입** | 웹 | URL만으로 즉시 시작 |
| **법률/의료 문서** | PDF | 정확한 형식 보존 필요 |
| **뉴스/블로그** | 웹 | 자주 업데이트되는 콘텐츠 |

### 주의사항

- ⚠️ **웹 크롤링 정책**: robots.txt 확인 및 이용 약관 준수
- ⚠️ **HTML 노이즈**: SoupStrainer로 본문만 추출
- ⚠️ **동적 콘텐츠**: JavaScript 렌더링이 필요한 경우 Selenium 사용 고려
- ⚠️ **인코딩**: 한글 깨짐 방지를 위한 인코딩 설정 확인

### 다음 단계

다음 노트북에서는 RAG 성능을 향상시키는 고급 기법들을 다룹니다:
- Ensemble Retriever (벡터 + 키워드 검색 결합)
- Reranking (검색 결과 재정렬)
- Query Transformation (질문 재작성)

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 도구 | `WebBaseLoader` + `BeautifulSoup` |
| HTML 필터링 | `SoupStrainer`로 원하는 태그(`div`, `p` 등)만 추출 |
| 로더 파라미터 | `bs_kwargs={"parse_only": SoupStrainer(...)}` |
| 실시간 정보 | URL 기반이므로 웹 콘텐츠가 갱신될 때마다 최신 정보 반영 |
| 주의 | 로그인이 필요하거나 JS 렌더링 페이지는 바로 사용 불가 |

### 웹 기반 RAG vs PDF 기반 RAG 비교

| 항목 | 웹 기반 | PDF 기반 |
|------|---------|----------|
| 로더 | `WebBaseLoader` | `PyPDFLoader` |
| 정보 최신성 | 실시간 | 파일 업로드 시점 |
| 접근 방식 | URL | 로컬 파일 경로 |
| HTML 파싱 | BeautifulSoup 필요 | 자동 처리 |
| 적합한 상황 | 뉴스, 위키, 공식 문서 | 보고서, 논문, 매뉴얼 |

### 다음 노트북 예고

**02-RAG-Advanced** — EnsembleRetriever(BM25+FAISS)와 MMR(최대 한계 관련성)을 결합해 기본 RAG보다 더 다양하고 관련성 높은 문서를 검색하는 방법을 배워요.